# Stage0/Stage1 KITTI Visualization (Range Image + PointPillar Boxes)

This notebook visualizes one KITTI validation frame with focused representative runs from **yesterday (2026-02-24)**:
- Stage0 baseline representative (uniform)
- Stage1 adaptive representative

For each representative run:
- Raw range image
- GT ROI label map (projected from KITTI labels)
- PointPillar detections on **raw** point cloud
- Decoded range image from compression model
- PointPillar detections on **reconstructed** point cloud

Important: PointPillar itself runs on point clouds. The range-image box overlay is produced by projecting each 3D box edge onto the same spherical range-image grid.



In [ ]:
import os
import sys
from pathlib import Path
import warnings
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if (ROOT / "notebooks").exists() and (ROOT / "src").exists():
    pass
else:
    ROOT = Path('/home/018219422/lidar_pointcloud_compression').resolve()

NOTEBOOK_DIR = ROOT / 'notebooks'
SRC_DIR = ROOT / 'src'
OPENPCDET_ROOT = ROOT / 'third_party' / 'OpenPCDet'
KITTI_ROOT = ROOT / 'data' / 'dataset' / 'kitti3dobject'
OPENPCDET_CFG = OPENPCDET_ROOT / 'tools' / 'cfgs' / 'kitti_models' / 'pointpillar.yaml'
OPENPCDET_CKPT = ROOT / 'data' / 'checkpoints' / 'openpcdet_pointpillar_18M.pth'

# Focused representatives from yesterday (2026-02-24):
# 1) Stage0 baseline representative
# 2) Stage1 representative (best among roi-target variants on yesterday runs)
RUN_SPECS = [
    {
        'tag': 'Stage0 Uniform Baseline (ResNet q8, j22255_r5)',
        'stage': '0',
        'run_dir': ROOT / 'data' / 'results' / 'experiments' / '260224_resnet_uniform_q8_lr1e-4_bs4_j22255_r5',
    },
    {
        'tag': 'Stage1 Adaptive ROI (ResNet nearest, j22279_r1)',
        'stage': '1',
        'run_dir': ROOT / 'data' / 'results' / 'experiments' / '260224_resnet_solo_lr1e-4_bs4_j22279_r1',
    },
]

for p in [SRC_DIR, OPENPCDET_ROOT, KITTI_ROOT, OPENPCDET_CFG, OPENPCDET_CKPT]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required path: {p}')
for spec in RUN_SPECS:
    if not spec['run_dir'].exists():
        raise FileNotFoundError(f"Missing run dir: {spec['run_dir']}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(OPENPCDET_ROOT) not in sys.path:
    sys.path.insert(0, str(OPENPCDET_ROOT))

# Ensure runtime can load torch/OpenPCDet shared libs in notebook kernels too.
torch_lib = Path(torch.__file__).resolve().parent / 'lib'
ld_parts = [str(torch_lib)]
nvhpc_root = Path('/opt/ohpc/pub/apps/nvidia/nvhpc/24.11/Linux_x86_64/24.11')
if nvhpc_root.exists():
    ld_parts.extend([
        str(nvhpc_root / 'cuda/11.8/lib64'),
        str(nvhpc_root / 'math_libs/11.8/targets/x86_64-linux/lib'),
    ])
old_ld = os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_parts + ([old_ld] if old_ld else []))

print('ROOT:', ROOT)
print('Torch:', torch.__version__, '| cuda_available=', torch.cuda.is_available())
print('Focused run count:', len(RUN_SPECS))
for s in RUN_SPECS:
    print('-', s['tag'])



In [ ]:
from dataset.kitti_object_loader import KittiObjectRangeDataset
from models.registry import MODELS
import models.compression  # noqa: F401
import models.backbones  # noqa: F401

from utils.recon_pointcloud_export import (
    reconstruct_kitti_points_with_model,
    project_points_to_range_image,
)

import train.evaluate_kitti_map_vs_rate as kmr

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for OpenPCDet inference. Run this notebook on a GPU node.')

DEVICE = torch.device('cuda')

# Optional: fix a specific sample manually (e.g., '000123').
SAMPLE_ID_OVERRIDE = None
TOPK_CANDIDATES = 60

# OpenPCDet detector (single reference model for raw/reconstructed comparison)
cfg, det_dataset, _, _, det_model, load_data_to_gpu = kmr._build_openpcdet_eval_objects(
    cfg_file=OPENPCDET_CFG,
    ckpt_file=OPENPCDET_CKPT,
    kitti_root=KITTI_ROOT,
    batch_size=1,
    workers=0,
)

def _norm_sid(sid):
    sid = str(sid)
    return sid.zfill(6) if sid.isdigit() else sid

def _target_obj_mask(names_obj):
    return np.isin(names_obj, ['Car', 'Pedestrian', 'Cyclist'])

def _proxy_candidate_score(info):
    ann = info.get('annos', {})
    names = np.asarray(ann.get('name', []))
    boxes_lidar = np.asarray(ann.get('gt_boxes_lidar', np.zeros((0, 7), dtype=np.float32)))
    bboxes_2d = np.asarray(ann.get('bbox', np.zeros((0, 4), dtype=np.float32)), dtype=np.float32)

    num_obj = int(boxes_lidar.shape[0])
    if num_obj <= 0:
        return -1.0

    names_obj = names[:num_obj] if names.size >= num_obj else names
    if names_obj.size == 0:
        return -1.0
    keep = _target_obj_mask(names_obj)
    if not np.any(keep):
        return -1.0

    if bboxes_2d.shape[0] >= num_obj:
        bb = bboxes_2d[:num_obj][keep]
    else:
        bb = bboxes_2d[keep[:bboxes_2d.shape[0]]]

    if bb.size == 0:
        area_sum = 0.0
    else:
        w = np.clip(bb[:, 2] - bb[:, 0], 0.0, None)
        h = np.clip(bb[:, 3] - bb[:, 1], 0.0, None)
        area_sum = float(np.sum(w * h))

    # object count + image bbox area proxy for larger ROI frames
    return float(2000.0 * np.sum(keep) + area_sum)

def _pick_sample_idx_with_large_roi(dataset, roi_dataset, sample_override=None, topk=60):
    sid_to_roi_idx = {sid: i for i, sid in enumerate(roi_dataset.sample_ids)}

    if sample_override is not None:
        sid = _norm_sid(sample_override)
        for i, info in enumerate(dataset.kitti_infos):
            cand_sid = _norm_sid(info['point_cloud']['lidar_idx'])
            if cand_sid == sid:
                if sid not in sid_to_roi_idx:
                    raise RuntimeError(f'Override sample {sid} not found in ROI dataset split.')
                roi_i = sid_to_roi_idx[sid]
                _, valid_t, roi_t = roi_dataset[roi_i]
                valid_np = valid_t.numpy() > 0.5
                roi_np = roi_t[0].numpy() > 0.5
                ratio = float(roi_np.sum() / max(valid_np.sum(), 1))
                return i, sid, ratio
        raise RuntimeError(f'Override sample {sid} not found in OpenPCDet val infos.')

    # 1) fast proxy ranking
    scored = []
    for i, info in enumerate(dataset.kitti_infos):
        sid = _norm_sid(info['point_cloud']['lidar_idx'])
        if sid not in sid_to_roi_idx:
            continue
        sc = _proxy_candidate_score(info)
        if sc > 0:
            scored.append((sc, i, sid))

    if not scored:
        raise RuntimeError('No candidate frames with target GT classes found.')

    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[: min(topk, len(scored))]

    # 2) exact ROI ratio on projected range mask for top candidates
    best = None
    for _, ds_i, sid in top:
        roi_i = sid_to_roi_idx[sid]
        _, valid_t, roi_t = roi_dataset[roi_i]
        valid_np = valid_t.numpy() > 0.5
        roi_np = roi_t[0].numpy() > 0.5
        ratio = float(roi_np.sum() / max(valid_np.sum(), 1))
        cnt = int(roi_np.sum())
        key = (ratio, cnt)
        if (best is None) or (key > (best[0], best[1])):
            best = (ratio, cnt, ds_i, sid)

    if best is None:
        raise RuntimeError('Failed to select large-ROI sample.')
    return best[2], best[3], float(best[0])

roi_ds = KittiObjectRangeDataset(root_dir=str(KITTI_ROOT), split='val', return_roi_mask=True)
sample_idx, sample_id, selected_roi_ratio = _pick_sample_idx_with_large_roi(
    det_dataset,
    roi_ds,
    sample_override=SAMPLE_ID_OVERRIDE,
    topk=TOPK_CANDIDATES,
)
print('Selected sample:', sample_id, '(dataset idx:', sample_idx, ')')
print(f'Selected sample ROI ratio on range image: {selected_roi_ratio:.4f}')



In [ ]:
EDGE_IDX = [
    (0, 1), (1, 2), (2, 3), (3, 0),
    (4, 5), (5, 6), (6, 7), (7, 4),
    (0, 4), (1, 5), (2, 6), (3, 7),
]

CLASS_COLOR = {
    'Car': '#00D9FF',
    'Pedestrian': '#FFB000',
    'Cyclist': '#7CFC00',
}
PRED_COLOR = '#FF3EB5'

def boxes_to_corners_3d_lidar(boxes):
    boxes = np.asarray(boxes, dtype=np.float32)
    if boxes.size == 0:
        return np.zeros((0, 8, 3), dtype=np.float32)
    centers = boxes[:, :3]
    dims = boxes[:, 3:6]
    yaws = boxes[:, 6]

    template = np.array([
        [ 1,  1, -1],
        [ 1, -1, -1],
        [-1, -1, -1],
        [-1,  1, -1],
        [ 1,  1,  1],
        [ 1, -1,  1],
        [-1, -1,  1],
        [-1,  1,  1],
    ], dtype=np.float32) * 0.5

    corners = dims[:, None, :] * template[None, :, :]
    c = np.cos(yaws)
    s = np.sin(yaws)
    rot = np.stack([
        np.stack([c, -s, np.zeros_like(c)], axis=-1),
        np.stack([s,  c, np.zeros_like(c)], axis=-1),
        np.stack([np.zeros_like(c), np.zeros_like(c), np.ones_like(c)], axis=-1),
    ], axis=1)
    corners = np.einsum('nij,nkj->nki', rot, corners)
    corners += centers[:, None, :]
    return corners

def project_xyz_to_range(xyz, H=64, W=1024, fov_up_deg=3.0, fov_down_deg=-25.0):
    xyz = np.asarray(xyz, dtype=np.float32)
    depth = np.linalg.norm(xyz, axis=1)
    valid = depth > 1e-4
    depth = np.clip(depth, 1e-4, None)

    yaw = -np.arctan2(xyz[:, 1], xyz[:, 0])
    pitch = np.arcsin(np.clip(xyz[:, 2] / depth, -1.0, 1.0))

    fov_up = np.deg2rad(fov_up_deg)
    fov_down = np.deg2rad(fov_down_deg)
    fov = fov_up - fov_down

    u = 0.5 * (yaw / np.pi + 1.0) * W
    v = (1.0 - (pitch + np.abs(fov_down)) / fov) * H
    inside = valid & (u >= 0) & (u < W) & (v >= 0) & (v < H)
    return u, v, inside

def boxes_to_range_points(boxes, H=64, W=1024, fov_up_deg=3.0, fov_down_deg=-25.0, samples_per_edge=60):
    corners = boxes_to_corners_3d_lidar(boxes)
    all_pts = []
    if corners.shape[0] == 0:
        return np.zeros((0, 2), dtype=np.float32)

    t = np.linspace(0.0, 1.0, samples_per_edge, dtype=np.float32)[:, None]
    for c in corners:
        for i0, i1 in EDGE_IDX:
            seg = (1.0 - t) * c[i0] + t * c[i1]
            u, v, ok = project_xyz_to_range(seg, H=H, W=W, fov_up_deg=fov_up_deg, fov_down_deg=fov_down_deg)
            if np.any(ok):
                all_pts.append(np.stack([u[ok], v[ok]], axis=1))

    if not all_pts:
        return np.zeros((0, 2), dtype=np.float32)
    return np.concatenate(all_pts, axis=0).astype(np.float32)

def plot_box_overlay(ax, boxes, color, label=None, s=1.0, alpha=0.9):
    pts = boxes_to_range_points(boxes)
    if pts.shape[0] == 0:
        return
    ax.scatter(pts[:, 0], pts[:, 1], s=s, c=color, alpha=alpha, label=label, linewidths=0)

def prediction_to_numpy(pred):
    out = {}
    for k, v in pred.items():
        if torch.is_tensor(v):
            out[k] = v.detach().cpu().numpy()
        else:
            out[k] = v
    return out

def run_detector_for_sample(dataset, sample_idx, detector_model, load_data_to_gpu, override_points=None):
    base_get_lidar = dataset.get_lidar
    sid = _norm_sid(dataset.kitti_infos[sample_idx]['point_cloud']['lidar_idx'])

    if override_points is not None:
        override_points = np.asarray(override_points, dtype=np.float32)
        def patched_get_lidar(sample_arg):
            if _norm_sid(sample_arg) == sid:
                return override_points
            return base_get_lidar(sample_arg)
        dataset.get_lidar = patched_get_lidar

    try:
        with torch.no_grad():
            data_dict = dataset[sample_idx]
            batch_dict = dataset.collate_batch([data_dict])
            if kmr._is_empty_voxel_batch(batch_dict):
                device = next(detector_model.parameters()).device
                pred = kmr._make_empty_pred_dicts(1, device)[0]
            else:
                load_data_to_gpu(batch_dict)
                pred_dicts, _ = detector_model(batch_dict)
                pred = pred_dicts[0]
    finally:
        dataset.get_lidar = base_get_lidar

    return prediction_to_numpy(pred)

def filter_predictions(pred, score_thr=0.30):
    boxes = np.asarray(pred.get('pred_boxes', np.zeros((0, 7), dtype=np.float32)), dtype=np.float32)
    scores = np.asarray(pred.get('pred_scores', np.zeros((0,), dtype=np.float32)), dtype=np.float32)
    labels = np.asarray(pred.get('pred_labels', np.zeros((0,), dtype=np.int64)), dtype=np.int64)
    if boxes.shape[0] == 0:
        return boxes, scores, labels
    keep = scores >= float(score_thr)
    return boxes[keep], scores[keep], labels[keep]


In [ ]:
# Prepare per-sample GT and raw detector prediction.

info = det_dataset.kitti_infos[sample_idx]
ann = info.get('annos', {})
all_gt_boxes = np.asarray(ann.get('gt_boxes_lidar', np.zeros((0, 7), dtype=np.float32)), dtype=np.float32)
all_gt_names = np.asarray(ann.get('name', []))

# In KITTI infos, name may include extra non-object entries (e.g., DontCare).
# gt_boxes_lidar corresponds to the first num_obj valid objects.
num_obj = int(all_gt_boxes.shape[0])
obj_names = all_gt_names[:num_obj] if all_gt_names.size >= num_obj else all_gt_names
keep_gt = np.isin(obj_names, ['Car', 'Pedestrian', 'Cyclist']) if obj_names.size else np.zeros((0,), dtype=bool)

GT_BOXES = all_gt_boxes[keep_gt] if num_obj > 0 else np.zeros((0, 7), dtype=np.float32)
GT_NAMES = obj_names[keep_gt] if obj_names.size else np.asarray([])

RAW_POINTS = det_dataset.get_lidar(sample_id).astype(np.float32)
RAW_PRED = run_detector_for_sample(det_dataset, sample_idx, det_model, load_data_to_gpu, override_points=None)
RAW_BOXES, RAW_SCORES, RAW_LABELS = filter_predictions(RAW_PRED, score_thr=0.30)

raw_5ch, raw_valid = project_points_to_range_image(RAW_POINTS)
RAW_RANGE = raw_5ch[0]

if 'roi_ds' not in globals():
    roi_ds = KittiObjectRangeDataset(root_dir=str(KITTI_ROOT), split='val', return_roi_mask=True)
roi_idx = roi_ds.sample_ids.index(sample_id)
_, _, roi_mask_t = roi_ds[roi_idx]
GT_ROI = roi_mask_t[0].numpy().astype(np.float32)

roi_ratio = float((GT_ROI > 0.5).sum() / max((raw_valid > 0.5).sum(), 1))

print('GT boxes:', GT_BOXES.shape[0])
print('Raw prediction boxes @0.30:', RAW_BOXES.shape[0])
print(f'GT ROI ratio (selected frame): {roi_ratio:.4f}')



In [ ]:
# Reconstruct with focused Stage0/Stage1 runs and run detector on reconstructed points.

RUN_RESULTS = []

for spec in RUN_SPECS:
    run_dir = spec['run_dir']
    tag = spec['tag']

    comp_model, comp_cfg, ckpt_path, uniform_bits = kmr._load_compression_model(run_dir, None, DEVICE)
    recon_points, rate_metrics, debug = reconstruct_kitti_points_with_model(
        model=comp_model,
        device=DEVICE,
        points_xyzi=RAW_POINTS,
        quantize=True,
        noise_std=0.0,
        img_h=64,
        img_w=1024,
        fov_up_deg=3.0,
        fov_down_deg=-25.0,
        range_threshold=1e-3,
        uniform_bits_fallback=uniform_bits,
    )

    recon_pred = run_detector_for_sample(
        det_dataset,
        sample_idx,
        det_model,
        load_data_to_gpu,
        override_points=recon_points,
    )
    recon_boxes, recon_scores, recon_labels = filter_predictions(recon_pred, score_thr=0.30)

    recon_range = debug['recon_5ch'][0]

    RUN_RESULTS.append({
        'tag': tag,
        'stage': spec.get('stage', '?'),
        'run_dir': str(run_dir),
        'ckpt_path': str(ckpt_path),
        'quantizer_mode': str(comp_cfg.get('model', {}).get('quantizer_config', {}).get('mode', 'unknown')),
        'roi_target_mode': str(comp_cfg.get('supervision', {}).get('roi_target_mode', 'unknown')),
        'raw_pred_count': int(RAW_BOXES.shape[0]),
        'recon_pred_count': int(recon_boxes.shape[0]),
        'recon_points_count': int(recon_points.shape[0]),
        'rate_metrics': rate_metrics,
        'recon_range': recon_range,
        'recon_boxes': recon_boxes,
        'recon_scores': recon_scores,
        'recon_labels': recon_labels,
    })

print('Collected focused results:', len(RUN_RESULTS))
for r in RUN_RESULTS:
    print(
        f"- {r['tag']} | mode={r['quantizer_mode']} | roi={r['roi_target_mode']} | "
        f"raw_pred={r['raw_pred_count']} -> recon_pred={r['recon_pred_count']} | "
        f"bpp_entropy={r['rate_metrics'].get('bpp_entropy', float('nan')):.4f}"
    )

if len(RUN_RESULTS) == 2:
    s0 = [r for r in RUN_RESULTS if str(r.get('stage', '')) == '0']
    s1 = [r for r in RUN_RESULTS if str(r.get('stage', '')) == '1']
    if s0 and s1:
        a = s0[0]
        b = s1[0]
        print('\nMeaningful comparison (yesterday representative):')
        print(f"- Stage0 baseline recon_pred={a['recon_pred_count']}, bpp_entropy={a['rate_metrics'].get('bpp_entropy', float('nan')):.4f}")
        print(f"- Stage1 adaptive recon_pred={b['recon_pred_count']}, bpp_entropy={b['rate_metrics'].get('bpp_entropy', float('nan')):.4f}")



In [ ]:
# Visualization: raw vs reconstructed + GT and PointPillar boxes.

from matplotlib.colors import ListedColormap, BoundaryNorm

range_valid_vals = RAW_RANGE[raw_valid > 0.5]
if range_valid_vals.size > 0:
    rmin, rmax = np.percentile(range_valid_vals, [2, 98])
else:
    rmin, rmax = 0.0, 80.0

# Explicit ROI palette to avoid white ROI confusion:
# no-data=dark gray, background=blue, ROI=yellow
roi_cmap = ListedColormap(['#202020', '#2F5D8A', '#FFD166'])
roi_norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], roi_cmap.N)

# Keep image geometry close to native range-image ratio (H:W ~= 64:1024).
img_box_aspect = float(RAW_RANGE.shape[0]) / float(RAW_RANGE.shape[1])

for res in RUN_RESULTS:
    recon = res['recon_range']
    err = np.abs(recon - RAW_RANGE)
    err_vmax = np.percentile(err[raw_valid > 0.5], 98) if (raw_valid > 0.5).any() else 1.0

    fig, axes = plt.subplots(3, 2, figsize=(36, 11), dpi=130)

    # Row 1: raw range + ROI label map
    im0 = axes[0, 0].imshow(np.where(raw_valid > 0.5, RAW_RANGE, np.nan), cmap='viridis', aspect='equal', vmin=rmin, vmax=rmax)
    axes[0, 0].set_title('Raw Range Image')
    plt.colorbar(im0, ax=axes[0, 0], fraction=0.018, pad=0.01)

    roi_vis = np.full(GT_ROI.shape, -1.0, dtype=np.float32)
    valid_mask = (raw_valid > 0.5)
    roi_vis[valid_mask] = 0.0
    roi_vis[valid_mask & (GT_ROI > 0.5)] = 1.0

    im1 = axes[0, 1].imshow(roi_vis, cmap=roi_cmap, norm=roi_norm, aspect='equal')
    axes[0, 1].set_title('GT ROI Label Map (No-data / BG / ROI)')
    cbar1 = plt.colorbar(im1, ax=axes[0, 1], fraction=0.018, pad=0.01, ticks=[-1, 0, 1])
    cbar1.ax.set_yticklabels(['No-data', 'BG', 'ROI'])

    # Row 2: raw with boxes + decoded range
    axes[1, 0].imshow(np.where(raw_valid > 0.5, RAW_RANGE, np.nan), cmap='viridis', aspect='equal', vmin=rmin, vmax=rmax)
    plot_box_overlay(axes[1, 0], GT_BOXES, color='#00D9FF', label='GT 3D box', s=1.2)
    plot_box_overlay(axes[1, 0], RAW_BOXES, color=PRED_COLOR, label='PointPillar(raw)', s=1.2)
    axes[1, 0].set_title(f"Raw Range + Boxes (pred={RAW_BOXES.shape[0]})")
    axes[1, 0].legend(loc='upper right', fontsize=8)

    im3 = axes[1, 1].imshow(np.where(raw_valid > 0.5, recon, np.nan), cmap='viridis', aspect='equal', vmin=rmin, vmax=rmax)
    axes[1, 1].set_title('Decoded Range Image')
    plt.colorbar(im3, ax=axes[1, 1], fraction=0.018, pad=0.01)

    # Row 3: error + decoded with boxes
    im4 = axes[2, 0].imshow(np.where(raw_valid > 0.5, err, np.nan), cmap='inferno', aspect='equal', vmin=0, vmax=err_vmax)
    axes[2, 0].set_title('Abs Error |decoded - raw|')
    plt.colorbar(im4, ax=axes[2, 0], fraction=0.018, pad=0.01)

    axes[2, 1].imshow(np.where(raw_valid > 0.5, recon, np.nan), cmap='viridis', aspect='equal', vmin=rmin, vmax=rmax)
    plot_box_overlay(axes[2, 1], GT_BOXES, color='#00D9FF', label='GT 3D box', s=1.2)
    plot_box_overlay(axes[2, 1], res['recon_boxes'], color=PRED_COLOR, label='PointPillar(recon)', s=1.2)
    axes[2, 1].set_title(f"Decoded Range + Boxes (pred={res['recon_boxes'].shape[0]})")
    axes[2, 1].legend(loc='upper right', fontsize=8)

    for ax in axes.reshape(-1):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_box_aspect(img_box_aspect)

    fig.suptitle(
        f"{res['tag']} | sample={sample_id} | "
        f"raw_pred={res['raw_pred_count']} | recon_pred={res['recon_pred_count']} | "
        f"bpp_entropy={res['rate_metrics'].get('bpp_entropy', float('nan')):.4f}",
        fontsize=14,
        y=1.01,
    )
    plt.tight_layout()
    plt.show()



## How the box appears on range image

- PointPillar predicts 3D boxes in LiDAR coordinates from point clouds.
- For range-image visualization only, each 3D box edge is sampled in 3D and projected with the same spherical projection used by the range image.
- So this overlay is a visualization transform, not a separate 2D detector.
